<a href="https://colab.research.google.com/github/IssarapongB/Data-science/blob/main/%E0%B8%9B%E0%B8%A3%E0%B8%B0%E0%B8%AA%E0%B8%9E%E0%B8%81%E0%B8%B2%E0%B8%A3%E0%B8%93%E0%B9%8C%E0%B8%A7%E0%B8%B4%E0%B8%8A%E0%B8%B2%E0%B8%8A%E0%B8%B5%E0%B8%9E1_68_5_Face_detect.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ===== 0) Install =====
!pip install -q deepface retina-face

# ===== 1) Imports =====
from deepface import DeepFace
from google.colab import files
import numpy as np
import cv2
import matplotlib.pyplot as plt

# ===== 2) Upload known face(s) =====
# Upload one or multiple single-person images (filenames become the person's name)
# e.g., "Alice.jpg", "Bob.png"
print("Upload one or more KNOWN face images (single face per image).")
known_files = files.upload()
known_paths = list(known_files.keys())

# ===== 3) Upload test image (group or single) =====
print("Upload a TEST image (can contain multiple faces).")
test_files = files.upload()
test_path = list(test_files.keys())[0]

# ===== 4) Build known embeddings (name → embedding) =====
def name_from_filename(path):
    # Strip extension; use filename as label
    import os
    base = os.path.basename(path)
    return os.path.splitext(base)[0]

model_name = "ArcFace"          # good accuracy/speed
detector_backend = "retinaface" # robust detection

known_db = []  # list of dicts: {"name": str, "embedding": vector}
for kp in known_paths:
    # represent() returns list of dicts (handle the first face)
    reps = DeepFace.represent(img_path=kp,
                              model_name=model_name,
                              detector_backend=detector_backend,
                              enforce_detection=True)
    if len(reps) == 0:
        print(f"[WARN] No face found in {kp}")
        continue
    known_db.append({
        "name": name_from_filename(kp),
        "embedding": reps[0]["embedding"]
    })

if not known_db:
    raise RuntimeError("No known faces encoded. Please upload at least one valid face image.")

# ===== 5) Detect faces in test image =====
faces = DeepFace.extract_faces(img_path=test_path, detector_backend=detector_backend, enforce_detection=False)

# Load original test image for annotation
orig = cv2.cvtColor(cv2.imread(test_path), cv2.COLOR_BGR2RGB)
annotated = orig.copy()

# ===== 6) Helpers =====
def parse_box(fa):
    """Return (x, y, w, h) from DeepFace 'facial_area' which can be dict or tuple/list."""
    if isinstance(fa, dict):
        x = fa.get('x', fa.get('left', 0))
        y = fa.get('y', fa.get('top', 0))
        if 'w' in fa: w = fa['w']
        elif 'width' in fa: w = fa['width']
        elif 'right' in fa: w = fa['right'] - x
        else: w = 0
        if 'h' in fa: h = fa['h']
        elif 'height' in fa: h = fa['height']
        elif 'bottom' in fa: h = fa['bottom'] - y
        else: h = 0
        return int(x), int(y), int(w), int(h)
    elif isinstance(fa, (list, tuple)) and len(fa) >= 4:
        x, y, w, h = fa[:4]
        return int(x), int(y), int(w), int(h)
    else:
        raise ValueError(f"Unexpected facial_area format: {type(fa)} → {fa}")

def cosine_sim(a, b):
    a = np.array(a); b = np.array(b)
    a = a / (np.linalg.norm(a) + 1e-12)
    b = b / (np.linalg.norm(b) + 1e-12)
    return float(np.dot(a, b))

# ===== 7) Recognize each detected face =====
# threshold: higher = stricter match; 0.35~0.45 is a reasonable start for ArcFace cosine similarity
THRESHOLD = 0.40

for f in faces:
    # 7.1 box
    x, y, w, h = parse_box(f['facial_area'])

    # 7.2 embedding from aligned face image (skip detection)
    rep = DeepFace.represent(
        img_path=f['face'],              # numpy array (aligned RGB face)
        model_name=model_name,
        detector_backend='skip',
        enforce_detection=False
    )[0]['embedding']

    # 7.3 compare to all knowns → best match
    best_name, best_sim = "Unknown", -1.0
    for person in known_db:
        sim = cosine_sim(person["embedding"], rep)
        if sim > best_sim:
            best_sim = sim
            best_name = person["name"]
    label = f"{best_name} ({best_sim:.2f})" if best_sim >= THRESHOLD else f"Unknown ({best_sim:.2f})"

    # 7.4 draw box + label
    cv2.rectangle(annotated, (x, y), (x + w, y + h), (0, 255, 0), 2)
    cv2.putText(annotated, label, (x, max(0, y - 10)),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)

# ===== 8) Show result =====
plt.figure(figsize=(10, 10))
plt.axis('off')
plt.imshow(annotated)
plt.show()

# ===== 9) Optional: save annotated image =====
cv2.imwrite("recognized.jpg", cv2.cvtColor(annotated, cv2.COLOR_RGB2BGR))
print("Saved annotated image to recognized.jpg")